# 🚒 London Fire Brigade — Préprocessing des données
**Liora Data Analyst Bootcamp | Les Mines PSL | Février 2026**
**Auteur : Gaëtan Seynave**

---

## Contexte
Ce notebook couvre l'intégralité du pipeline de préparation des données du projet LFB :
chargement des fichiers bruts, harmonisation, nettoyage, enrichissement et export
vers les fichiers `.parquet` utilisés par le tableau de bord Power BI.

**Sources :** [London Data Store](https://data.london.gov.uk/) — données publiques LFB
**Période analysée :** 1er janvier 2021 – 31 décembre 2025 (post-COVID, années pleines)

---

## Structure du notebook

| Étape | Contenu |
|---|---|
| 0 | Imports & fonctions utilitaires |
| 1 | Chargement et conversion en Parquet |
| 2 | Incidents — analyse, harmonisation, concaténation |
| 3 | Mobilisations — analyse, harmonisation, concaténation |
| 4 | Restriction temporelle 2021–2025 |
| 5 | Traitement des NaN — Incidents |
| 6 | Traitement des NaN — Mobilisations |
| 7 | Enrichissement météo (API Open-Meteo) |
| 8 | Exports finaux |

## 0. Imports & fonctions utilitaires

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyproj import Transformer
import openmeteo_requests
import requests_cache
from retry_requests import retry

%matplotlib inline
pd.set_option('display.max_columns', None)

### Fonctions utilitaires
Trois fonctions réutilisables pour l'exploration et la comparaison des DataFrames.

In [ ]:
def resume_df(df):
    """Résume les caractéristiques principales d'un DataFrame :
    type, nb valeurs uniques, nb et % de valeurs nulles."""
    resume = pd.DataFrame({
        "type": df.dtypes,
        "nb_valeurs_uniques": df.nunique(),
        "nb_valeurs_nulles": df.isna().sum(),
        "pourcentage_null": round((df.isna().sum() / len(df)) * 100, 2)
    })
    return resume.sort_values("nb_valeurs_uniques")


def compare_columns(dfs: dict):
    """Compare les noms de colonnes entre plusieurs DataFrames.
    Retourne les colonnes communes et les différences."""
    cols = {k: set(v.columns) for k, v in dfs.items()}
    common = None
    union = set()
    for col_set in cols.values():
        common = col_set.copy() if common is None else common & col_set
        union |= col_set
    differences = {
        k: {
            "colonnes_specifiques": cols[k] - common,
            "colonnes_manquantes": union - cols[k]
        }
        for k in dfs
    }
    return {"colonnes_communes": common, "colonnes_differentes": differences}


def compare_dtypes(dfs: dict):
    """Compare les types de données des colonnes communes entre plusieurs DataFrames.
    Affiche uniquement les colonnes présentant des différences."""
    dtype_table = pd.DataFrame({k: v.dtypes.astype(str) for k, v in dfs.items()})
    diffs = dtype_table[dtype_table.nunique(axis=1) > 1].sort_index()
    print(f"Colonnes avec dtypes différents : {diffs.shape[0]}")
    return diffs


def iqr_outlier_detection(df, feature, threshold=1.5):
    """Détecte les outliers d'une colonne via la méthode IQR.

    Args:
        df: DataFrame source
        feature: nom de la colonne à analyser
        threshold: multiplicateur IQR (défaut 1.5)
    Returns:
        DataFrame des lignes identifiées comme outliers
    """
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - threshold * iqr
    upper_bound = q3 + threshold * iqr
    outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
    proportion = (len(outliers) / len(df)) * 100
    stats = pd.DataFrame({
        "Proportion d'outliers (%)": [round(proportion, 2)],
        "Nombre d'outliers": [len(outliers)]
    })
    print(f"Statistiques pour la colonne : {feature}")
    display(stats)
    return outliers

## 1. Chargement et conversion en Parquet

Les fichiers sources (`.csv` et `.xlsx`) sont convertis en `.parquet` pour optimiser
les performances de lecture (poids réduit, types préservés, calculs plus rapides).

> ⚠️ **Données sources non incluses dans ce repository** (fichiers > 500 Mo).
> Téléchargeables gratuitement sur le [London Data Store](https://data.london.gov.uk/) :
> - [Incident Records](https://data.london.gov.uk/dataset/london-fire-brigade-incident-records-em8xy/)
> - [Mobilisation Records](https://data.london.gov.uk/dataset/london-fire-brigade-mobilisation-records-24r65/)
>
> Placer les fichiers dans le dossier `data/raw/` avant d'exécuter cette cellule.

In [ ]:
dossier = os.path.join(os.getcwd(), 'data', 'raw')

fichiers_a_convertir = [
    'LFB Incident data from 2009 - 2017.csv',
    'LFB Incident data from 2018 - 2023.xlsx',
    'LFB Incident data from 2024 onwards.xlsx',
    'LFB Mobilisation data from January 2009 - 2014.xlsx',
    'LFB Mobilisation data from 2015 - 2020.xlsx',
    'LFB Mobilisation data from 2021 - 2024.csv',
    'LFB Mobilisation data from 2025.csv'
]

for fichier in fichiers_a_convertir:
    chemin_complet = os.path.join(dossier, fichier)
    nouveau_nom = fichier.replace('.xlsx', '.parquet').replace('.csv', '.parquet')
    chemin_parquet = os.path.join(dossier, nouveau_nom)
    print(f"Lecture : {fichier}...")
    if fichier.endswith('.xlsx'):
        df = pd.read_excel(chemin_complet)
    elif fichier.endswith('.csv'):
        df = pd.read_csv(chemin_complet, low_memory=False)
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str)
    df.to_parquet(chemin_parquet, index=False)
    print(f"✅ Sauvegardé : {nouveau_nom}\n")

print("Toutes les conversions en Parquet sont terminées.")

## 2. Incidents — Analyse, harmonisation, concaténation

Les données incidents sont réparties sur **3 fichiers** couvrant 2009–2026.
Avant concaténation : vérification des colonnes et uniformisation des types.

In [ ]:
incident_1 = pd.read_parquet(os.path.join(dossier, 'LFB Incident data from 2009 - 2017.parquet'))
incident_2 = pd.read_parquet(os.path.join(dossier, 'LFB Incident data from 2018 - 2023.parquet'))
incident_3 = pd.read_parquet(os.path.join(dossier, 'LFB Incident data from 2024 onwards.parquet'))

dfsinc = {"inc1": incident_1, "inc2": incident_2, "inc3": incident_3}

### 2.1 Comparaison des colonnes

In [ ]:
compare_columns(dfsinc)

> ✅ Les 3 fichiers partagent exactement les mêmes colonnes — aucun ajustement nécessaire.

### 2.2 Uniformisation des types de données

In [ ]:
compare_dtypes(dfsinc)

> 3 colonnes présentent des types hétérogènes entre les fichiers → uniformisation ci-dessous.

In [ ]:
incident_1['DateOfCall'] = pd.to_datetime(incident_1['DateOfCall'], format='%d-%b-%y')

for df in [incident_1, incident_2, incident_3]:
    df['UPRN'] = df['UPRN'].astype('Int64')
    df['USRN'] = df['USRN'].astype('Int64')

### 2.3 Concaténation

In [ ]:
incident_complet = pd.concat([incident_1, incident_2, incident_3], ignore_index=True)
incident_complet = incident_complet.drop_duplicates()

incident_complet['IncidentWithMobilisation'] = (
    incident_complet['FirstPumpArriving_AttendanceTime'].notna().astype(int)
)

print(f"Dimensions du DataFrame incidents complet : {incident_complet.shape}")
incident_complet.head(3)

## 3. Mobilisations — Analyse, harmonisation, concaténation

Les données mobilisations sont réparties sur **4 fichiers** couvrant 2009–2025.

In [ ]:
mobilisation_1 = pd.read_parquet(os.path.join(dossier, 'LFB Mobilisation data from January 2009 - 2014.parquet'))
mobilisation_2 = pd.read_parquet(os.path.join(dossier, 'LFB Mobilisation data from 2015 - 2020.parquet'))
mobilisation_3 = pd.read_parquet(os.path.join(dossier, 'LFB Mobilisation data from 2021 - 2024.parquet'))
mobilisation_4 = pd.read_parquet(os.path.join(dossier, 'LFB Mobilisation data from 2025.parquet'))

dfsmob = {"mob1": mobilisation_1, "mob2": mobilisation_2,
          "mob3": mobilisation_3, "mob4": mobilisation_4}

In [ ]:
compare_columns(dfsmob)

In [ ]:
compare_dtypes(dfsmob)

> 6 colonnes présentent des types hétérogènes → uniformisation ci-dessous.

In [ ]:
cols_dates = ['DateAndTimeArrived', 'DateAndTimeLeft', 'DateAndTimeMobile',
              'DateAndTimeMobilised', 'DateAndTimeReturned']

for col in cols_dates:
    for df in [mobilisation_3, mobilisation_4]:
        df[col] = pd.to_datetime(df[col], format="%d/%m/%Y %H:%M")

for df in [mobilisation_1, mobilisation_3, mobilisation_4]:
    df['IncidentNumber'] = df['IncidentNumber'].astype('object')

### 3.3 Concaténation & nettoyage de PerformanceReporting

In [ ]:
mobilisation_complet = pd.concat(
    [mobilisation_1, mobilisation_2, mobilisation_3, mobilisation_4],
    ignore_index=True
)
mobilisation_complet = mobilisation_complet.drop_duplicates()

s = mobilisation_complet['PerformanceReporting']
mobilisation_complet['PerformanceReporting'] = (
    pd.to_numeric(
        s.replace("Not Used", "0").astype(str).str.strip(),
        errors="coerce"
    )
    .fillna(0)
    .astype("Int64")
)

print(f"Dimensions du DataFrame mobilisations complet : {mobilisation_complet.shape}")
mobilisation_complet.head(3)

## 4. Restriction temporelle : 2021–2025

**Choix méthodologique :** la période est limitée à janvier 2021 – décembre 2025 pour :
- Exclure la période COVID (dernier confinement : décembre 2020)
- Travailler sur des années pleines comparables
- Disposer des colonnes `BoroughName` et `WardName` dans les mobilisations (disponibles à partir de 2021)

In [ ]:
start_study = '2021-01-01'
end_study   = '2025-12-31 23:59:59'

incident_postcovid = incident_complet.loc[
    (incident_complet['DateOfCall'] >= start_study) &
    (incident_complet['DateOfCall'] <= end_study)
].copy()

new_types = {
    'NumStationsWithPumpsAttending': 'Int64',
    'NumPumpsAttending': 'Int64',
    'NumCalls': 'Int64'
}
incident_postcovid = incident_postcovid.astype(new_types)

mobilisation_postcovid = mobilisation_complet.loc[
    (mobilisation_complet['DateAndTimeMobilised'] >= start_study) &
    (mobilisation_complet['DateAndTimeMobilised'] <= end_study)
].copy()
mobilisation_postcovid = mobilisation_postcovid.drop_duplicates()

print(f"✅ Incidents     : {len(incident_postcovid):,} lignes")
print(f"✅ Mobilisations : {len(mobilisation_postcovid):,} lignes")

## 5. Traitement des valeurs manquantes — Incidents

### 5.1 Visualisation globale des NaN

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(incident_postcovid.isna(), yticklabels=False,
            cbar=False, cmap="viridis", ax=axes[0])
axes[0].set_title("Incidents : valeurs manquantes (global)", fontsize=13)

nan_par_an = incident_postcovid.isna().groupby(
    incident_postcovid['CalYear']).mean() * 100
sns.heatmap(nan_par_an.T, annot=True, fmt=".1f", cmap="YlOrRd",
            cbar_kws={'label': '% manquant'}, ax=axes[1])
axes[1].set_title("Taux de NaN par année et par variable", fontsize=13)

plt.tight_layout()
plt.show()

> **Décisions de traitement des NaN :**
>
> | Colonne | NaN | Décision |
> |---|---|---|
> | `SpecialServiceType` | ~388k | Fusionnée dans `StopCodeDescription` puis supprimée |
> | `Postcode_full` / `Easting_m` / `Latitude` / `Longitude` | ~375k | Supprimées (anonymisées par la LFB) |
> | `FirstPumpArriving_AttendanceTime` | ~35k | Conservée — NaN = appel sans envoi de camion |
> | `SecondPumpArriving_AttendanceTime` | ~405k | Conservée — NaN = pas de 2ème véhicule |

### 5.2 Fusion StopCodeDescription / SpecialServiceType

In [ ]:
masque_special = incident_postcovid['IncidentGroup'] == 'Special Service'
incident_postcovid.loc[masque_special, 'StopCodeDescription'] = (
    incident_postcovid.loc[masque_special, 'SpecialServiceType']
)
print("✅ Fusion StopCodeDescription / SpecialServiceType effectuée")

### 5.3 Conversion des coordonnées (British National Grid → WGS84)

Les coordonnées `Easting_rounded` / `Northing_rounded` sont en système britannique
(EPSG:27700). Conversion en latitude/longitude standard (EPSG:4326) pour Power BI.

In [ ]:
transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326")

def convert_to_latlon(east, north):
    """Convertit des coordonnées British National Grid en WGS84 (lat, lon)."""
    return transformer.transform(east, north)

incident_postcovid['latitude_rounded'], incident_postcovid['longitude_rounded'] = zip(
    *incident_postcovid.apply(
        lambda row: convert_to_latlon(row['Easting_rounded'], row['Northing_rounded']),
        axis=1
    )
)
print("✅ Conversion coordonnées terminée")
incident_postcovid[['Easting_rounded', 'Northing_rounded',
                     'latitude_rounded', 'longitude_rounded']].head(3)

### 5.4 Suppression des colonnes inutiles

In [ ]:
columns_to_drop = [
    'ProperCase',         # Doublon de IncGeo_BoroughName
    'Easting_m',          # Anonymisé par LFB (précision au mètre)
    'Northing_m',         # Anonymisé par LFB (précision au mètre)
    'FRS',                # Valeur identique sur toute la colonne
    'SpecialServiceType', # Fusionnée dans StopCodeDescription
    'Latitude',           # Anonymisée par LFB
    'Longitude',          # Anonymisée par LFB
    'UPRN',               # Anonymisé par LFB
    'Postcode_full'       # Anonymisé par LFB
]

incident_postcovid.drop(columns=columns_to_drop, inplace=True)
print(f"✅ Colonnes restantes : {incident_postcovid.shape[1]}")

plt.figure(figsize=(15, 6))
sns.heatmap(incident_postcovid.isna(), yticklabels=False, cbar=False, cmap="viridis")
plt.title("Incidents après nettoyage : valeurs manquantes résiduelles", fontsize=13)
plt.show()

## 6. Traitement des valeurs manquantes — Mobilisations

### 6.1 Visualisation & analyse des NaN

In [ ]:
plt.figure(figsize=(15, 6))
sns.heatmap(mobilisation_postcovid.isna(), yticklabels=False, cbar=False, cmap="viridis")
plt.title("Mobilisations : valeurs manquantes (global)", fontsize=13)
plt.show()

resume_df(mobilisation_postcovid)

### 6.2 Traitement de DelayCode

**Analyse :** Les NaN dans `DelayCodeId` et `DelayCode_Description` correspondent
aux mobilisations **sans retard** (temps d'arrivée < 6 min pour le véhicule 1,
< 8 min pour le véhicule 2). Point confirmé par contact direct avec la LFB.

→ Les NaN sont remplacés par des valeurs sentinelles : `-1` et `'Sans délai'`.

In [ ]:
mobilisation_postcovid['DelayCodeId'] = (
    pd.to_numeric(mobilisation_postcovid['DelayCodeId'], errors='coerce')
    .fillna(-1)
    .astype('Int64')
)

mobilisation_postcovid['DelayCode_Description'] = (
    mobilisation_postcovid['DelayCode_Description']
    .astype(str)
    .replace(['nan', 'None', 'None '], 'Sans délai')
)

print("✅ DelayCode nettoyé")
mobilisation_postcovid['DelayCode_Description'].value_counts().head(10)

### 6.3 Création des indicateurs de performance légale

La LFB est soumise à des objectifs formalisés dans le **London Safety Plan** :
- 🎯 Véhicule 1 : arrivée en **≤ 6 minutes**
- 🎯 Véhicule 2 : arrivée en **≤ 8 minutes**

In [ ]:
mobilisation_postcovid['is_inferieur_6_min'] = np.where(
    mobilisation_postcovid['AttendanceTimeSeconds'] <= 6 * 60, 1, 0
)
mobilisation_postcovid['is_inferieur_8_min'] = np.where(
    mobilisation_postcovid['AttendanceTimeSeconds'] <= 8 * 60, 1, 0
)

print("✅ Indicateurs légaux créés")
print(f"Taux de respect objectif 6 min : "
      f"{mobilisation_postcovid['is_inferieur_6_min'].mean()*100:.1f}%")
print(f"Taux de respect objectif 8 min : "
      f"{mobilisation_postcovid['is_inferieur_8_min'].mean()*100:.1f}%")

### 6.4 Imputation des temps manquants (TurnoutTime, TravelTime, DateAndTimeMobile)

**Stratégie :** remplacement des NaN par la **moyenne de la caserne** concernée.
Si une caserne n'a aucune donnée → moyenne globale.

In [ ]:
for col in ['TurnoutTimeSeconds', 'TravelTimeSeconds']:
    mobilisation_postcovid[col] = pd.to_numeric(
        mobilisation_postcovid[col], errors='coerce'
    ).astype(float)

for col in ['TurnoutTimeSeconds', 'TravelTimeSeconds']:
    mobilisation_postcovid[col] = mobilisation_postcovid[col].fillna(
        mobilisation_postcovid.groupby('DeployedFromStation_Name')[col].transform('mean')
    )
    mobilisation_postcovid[col] = mobilisation_postcovid[col].fillna(
        mobilisation_postcovid[col].mean()
    )

mobilisation_postcovid['AttendanceTimeSeconds'] = (
    mobilisation_postcovid['TurnoutTimeSeconds'] +
    mobilisation_postcovid['TravelTimeSeconds']
)

mobilisation_postcovid['DateAndTimeMobilised'] = pd.to_datetime(
    mobilisation_postcovid['DateAndTimeMobilised'])
mobilisation_postcovid['DateAndTimeMobile'] = pd.to_datetime(
    mobilisation_postcovid['DateAndTimeMobile'])

mask_nan = mobilisation_postcovid['DateAndTimeMobile'].isna()
mobilisation_postcovid.loc[mask_nan, 'DateAndTimeMobile'] = (
    mobilisation_postcovid.loc[mask_nan, 'DateAndTimeMobilised'] +
    pd.to_timedelta(
        mobilisation_postcovid.loc[mask_nan, 'TurnoutTimeSeconds'].fillna(0).round(),
        unit='s'
    )
)

for col in ['TurnoutTimeSeconds', 'TravelTimeSeconds', 'AttendanceTimeSeconds']:
    mobilisation_postcovid[col] = mobilisation_postcovid[col].round().astype('Int64')

print("✅ Imputation des temps terminée")

## 7. Enrichissement météo (API Open-Meteo)

Données météo horaires récupérées via l'API **Open-Meteo Historical Archive** pour
**5 zones géographiques** de Londres sur 2021–2025.

> API gratuite, sans clé requise. Documentation : https://open-meteo.com/

In [ ]:
mapping_zones_codes = {
    'E09000001': 'Central', 'E09000007': 'Central', 'E09000019': 'Central',
    'E09000021': 'Central', 'E09000028': 'Central', 'E09000033': 'Central',
    'E09000003': 'North',   'E09000010': 'North',   'E09000014': 'North',
    'E09000017': 'North',   'E09000031': 'North',
    'E09000005': 'South',   'E09000008': 'South',   'E09000020': 'South',
    'E09000022': 'South',   'E09000024': 'South',   'E09000029': 'South',
    'E09000032': 'South',
    'E09000002': 'East',    'E09000004': 'East',    'E09000011': 'East',
    'E09000015': 'East',    'E09000016': 'East',    'E09000023': 'East',
    'E09000025': 'East',    'E09000026': 'East',    'E09000027': 'East',
    'E09000030': 'East',
    'E09000006': 'West',    'E09000009': 'West',    'E09000012': 'West',
    'E09000013': 'West',    'E09000018': 'West'
}
incident_postcovid['Zone'] = incident_postcovid['IncGeo_BoroughCode'].map(mapping_zones_codes)

zones_coords = {
    "Central": {"lat": 51.5074, "lon": -0.1278},
    "North":   {"lat": 51.6156, "lon": -0.1323},
    "South":   {"lat": 51.3762, "lon": -0.0982},
    "East":    {"lat": 51.5302, "lon":  0.0814},
    "West":    {"lat": 51.5042, "lon": -0.4483}
}

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

all_weather_data = []
for zone_name, coords in zones_coords.items():
    print(f"Extraction météo : {zone_name}...")
    params = {
        "latitude": coords["lat"], "longitude": coords["lon"],
        "start_date": "2021-01-01", "end_date": "2025-12-31",
        "hourly": ["temperature_2m", "relative_humidity_2m", "precipitation",
                   "wind_speed_10m", "wind_gusts_10m", "shortwave_radiation"],
        "timezone": "Europe/London"
    }
    responses = openmeteo.weather_api(
        "https://archive-api.open-meteo.com/v1/archive", params=params)
    res = responses[0]
    hourly = res.Hourly()
    df_temp = pd.DataFrame({
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()), inclusive="left"
        ),
        "temp_c": hourly.Variables(0).ValuesAsNumpy(),
        "humidity": hourly.Variables(1).ValuesAsNumpy(),
        "precip_mm": hourly.Variables(2).ValuesAsNumpy(),
        "wind_speed": hourly.Variables(3).ValuesAsNumpy(),
        "wind_gusts": hourly.Variables(4).ValuesAsNumpy(),
        "radiation": hourly.Variables(5).ValuesAsNumpy(),
        "Zone": zone_name
    })
    all_weather_data.append(df_temp)

df_weather_final = pd.concat(all_weather_data, ignore_index=True)
print(f"\n✅ Données météo : {df_weather_final.shape}")
df_weather_final.head(3)

## 8. Exports finaux

Export des DataFrames nettoyés en `.parquet` dans `data/processed/`
pour chargement dans Power BI.

In [ ]:
dossier_export = os.path.join(os.getcwd(), 'data', 'processed')
os.makedirs(dossier_export, exist_ok=True)

exports = {
    'incident_postcovid.parquet':             incident_postcovid,
    'mobilisation_postcovid.parquet':         mobilisation_postcovid,
    'london_weather_zones_2021_2025.parquet': df_weather_final
}

for nom_fichier, df in exports.items():
    chemin = os.path.join(dossier_export, nom_fichier)
    df.to_parquet(chemin, index=False)
    print(f"✅ Exporté : {nom_fichier} ({len(df):,} lignes)")

print("\n🎯 Preprocessing terminé — fichiers prêts pour Power BI")